In [ ]:
import re
import string
import pickle
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
for resource, pkg in [
    ("tokenizers/punkt", "punkt"),
    ("tokenizers/punkt_tab", "punkt_tab"),
    ("corpora/stopwords", "stopwords"),
]:
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(pkg, quiet=True)
from nltk.corpus import stopwords as nltk_stopwords
from nltk.tokenize import word_tokenize

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED   = 42
print(f"Device: {DEVICE}")

: 

## Cell 2 — Load Saved Artifacts

In [ ]:
MODEL_PATH    = Path("../saved_model.pth")
WORD2IDX_PATH = Path("../models/word2idx.pkl")
LABEL_PATH    = Path("../models/label_encoder.pkl")

for p in [MODEL_PATH, WORD2IDX_PATH, LABEL_PATH]:
    assert p.exists(), f"Missing: {p} — run train.ipynb first!"

with open(WORD2IDX_PATH, "rb") as f:
    WORD2IDX: Dict[str, int] = pickle.load(f)

with open(LABEL_PATH, "rb") as f:
    enc = pickle.load(f)
    STR2IDX: Dict[str, int] = enc["str2idx"]
    IDX2STR: Dict[int, str] = enc["idx2str"]

VOCAB_SIZE  = len(WORD2IDX)
NUM_CLASSES = len(STR2IDX)
CLASS_NAMES = [IDX2STR[i] for i in range(NUM_CLASSES)]
PAD_IDX, UNK_IDX = 0, 1

print(f"✅ Loaded vocabulary : {VOCAB_SIZE:,} words")
print(f"✅ Classes           : {CLASS_NAMES}")

## Cell 3 — Rebuild Model & Load Weights

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden_size=128,
                 num_layers=2, num_classes=5, lstm_dropout=0.5,
                 fc_dropout=0.3, bidirectional=True, pad_idx=0):
        super().__init__()
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_size, num_layers=num_layers,
                            bidirectional=bidirectional, batch_first=True,
                            dropout=lstm_dropout if num_layers > 1 else 0.0)
        dirs = 2 if bidirectional else 1
        self.dropout1 = nn.Dropout(lstm_dropout)
        self.fc1      = nn.Linear(hidden_size * dirs, hidden_size)
        self.relu     = nn.ReLU()
        self.dropout2 = nn.Dropout(fc_dropout)
        self.fc2      = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        _, (h, _) = self.lstm(emb)
        h = torch.cat((h[-2], h[-1]), dim=1) if self.bidirectional else h[-1]
        return self.fc2(self.dropout2(self.relu(self.fc1(self.dropout1(h)))))


model = BiLSTMClassifier(VOCAB_SIZE, num_classes=NUM_CLASSES).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()
print("✅ Model loaded and set to eval mode.")

## Cell 4 — Rebuild Test DataLoader

In [ ]:
STOP_WORDS = set(nltk_stopwords.words("english"))
MAX_SEQ_LEN = 512
BATCH_SIZE  = 64

LABEL_MAP = {
    "business": "Business", "entertainment": "Entertainment",
    "politics": "Politics", "sport": "Sports", "tech": "Technology",
}

def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", "", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\d+", "", text)
    return re.sub(r"\s+", " ", text).strip()

def preprocess(text: str) -> List[str]:
    tokens = word_tokenize(clean_text(text))
    return [t for t in tokens if t not in STOP_WORDS and len(t) > 1]

def text_to_indices(tokens: List[str], max_len: int) -> List[int]:
    indices = [WORD2IDX.get(t, UNK_IDX) for t in tokens[:max_len]]
    return indices + [PAD_IDX] * (max_len - len(indices))

class NewsDataset(Dataset):
    def __init__(self, texts, labels):
        self.labels    = labels
        self.sequences = [text_to_indices(preprocess(t), MAX_SEQ_LEN) for t in texts]
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return torch.tensor(self.sequences[i], dtype=torch.long), torch.tensor(self.labels[i], dtype=torch.long)

# Reload the dataset and rebuild the same test split
df = pd.read_csv("../data/bbc-text.csv")
df.columns = [c.strip().lower() for c in df.columns]
df["label"] = df["category"].map(LABEL_MAP)
df.dropna(subset=["text", "label"], inplace=True)
df["encoded_label"] = df["label"].map(STR2IDX)
df.reset_index(drop=True, inplace=True)

_, temp_df = train_test_split(df, test_size=0.20, stratify=df["encoded_label"], random_state=SEED)
_, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["encoded_label"], random_state=SEED)
test_df.reset_index(drop=True, inplace=True)

print("⏳ Building test dataset ...")
test_loader = DataLoader(NewsDataset(test_df["text"].tolist(), test_df["encoded_label"].tolist()),
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"✅ Test set: {len(test_df)} articles, {len(test_loader)} batches.")

## Cell 5 — Compute All Metrics

In [ ]:
@torch.no_grad()
def get_predictions(model, loader):
    all_preds, all_labels = [], []
    for seqs, labels in loader:
        seqs = seqs.to(DEVICE)
        preds = model(seqs).argmax(dim=1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels.tolist())
    return all_preds, all_labels

preds, labels = get_predictions(model, test_loader)

acc  = accuracy_score(labels, preds)
prec = precision_score(labels, preds, average="macro")
rec  = recall_score(labels, preds, average="macro")
f1   = f1_score(labels, preds, average="macro")

print(f"\n{'='*50}")
print(f"  TEST SET EVALUATION")
print(f"{'='*50}")
print(f"  Accuracy  : {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Precision : {prec:.4f}")
print(f"  Recall    : {rec:.4f}")
print(f"  F1-Score  : {f1:.4f}")
print(f"{'='*50}\n")
print("📋 Per-class Report:")
print(classification_report(labels, preds, target_names=CLASS_NAMES, digits=4))

## Cell 6 — Confusion Matrix

In [ ]:
cm     = confusion_matrix(labels, preds)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm_pct, annot=True, fmt=".1f",
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    cmap="Blues", linewidths=0.5, ax=ax
)
for i in range(len(CLASS_NAMES)):
    for j in range(len(CLASS_NAMES)):
        ax.text(j+0.5, i+0.72, f"({cm[i,j]})",
                ha="center", va="center", fontsize=8, color="gray")
ax.set_title("Confusion Matrix — Test Set (% row-normalised)", fontsize=12, fontweight="bold")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
Path("../models").mkdir(parents=True, exist_ok=True)
plt.savefig("../models/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("💾 Confusion matrix saved to ../models/confusion_matrix.png")